# 12. Sensibilidade da solução aos parâmetros

Validação usando `nucleo` e `mercado` já prontos. **F12.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
from app import nucleo
from app.mercado import RendaFixa, RendaVariavel

Os dados são uma série sintética mensal. Os cenários são sorteados uma vez só e usados em todas as reotimizações, assim a diferença entre um alpha* e outro vem do parâmetro que mudou, e não do sorteio.

In [2]:
rng = np.random.default_rng(7)
ruido = rng.normal(0,0.06,300)
ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=300,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido})
mkt = RendaVariavel(ret)
rf = 1 + RendaFixa(0.10).retorno_livre_risco()
r = mkt.amostrar(200_000, seed=1)

def alpha_de(cenarios, g=5.0):
    """alpha* sobre os cenarios (retornos liquidos), com o mesmo corte em zero da esteira."""
    return nucleo.resolver_alpha_otimo(np.maximum(1 + cenarios, 0).reshape(-1, 1), rf, g)[0]

**Teste**: o alpha* tem que cair quando o gamma sobe. É um resultado de estática comparativa: com um ativo arriscado só, quem é mais avesso ao risco aplica menos nele, qualquer que seja a distribuição dos retornos.

In [3]:
GAMMAS = (1.5, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0)
alpha_gamma = [alpha_de(r, g) for g in GAMMAS]

print('alpha* por gamma:', np.round(alpha_gamma, 4))

alpha* por gamma: [1.4988 1.1277 0.7535 0.4526 0.283  0.2264 0.151  0.1132]


In [4]:
assert np.all(np.diff(alpha_gamma) < 0)

**Teste**: o alpha* tem que subir com a média mu e cair com a volatilidade sigma, como na aproximação de média-variância alpha* aprox (mu - R_f)/(gamma * sigma^2). Para mudar mu eu desloco os cenários, e para mudar sigma eu estico eles em torno da média.

In [5]:
DESLOCAMENTOS = (-0.004, -0.002, 0.0, 0.002, 0.004)
alpha_mu = [alpha_de(r + d) for d in DESLOCAMENTOS]
media = r.mean()
ESCALAS = (0.6, 0.8, 1.0, 1.2, 1.4)
alpha_sigma = [alpha_de(media + (r - media) * k) for k in ESCALAS]

print('alpha* por deslocamento de mu:', np.round(alpha_mu, 4))
print('alpha* por escala de sigma   :', np.round(alpha_sigma, 4))

alpha* por deslocamento de mu: [0.1899 0.3213 0.4526 0.5837 0.7146]
alpha* por escala de sigma   : [1.2554 0.7069 0.4526 0.3144 0.231 ]


In [6]:
assert np.all(np.diff(alpha_mu) > 0) and np.all(np.diff(alpha_sigma) < 0)

**Teste**: com beta maior o investidor é mais paciente, então o theta_t tem que cair em todo t < T, e em t = T continua dando 1. Que o alpha* não muda com o beta não precisa de teste, porque o beta nem entra na G.

In [7]:
R = np.maximum(1 + r, 0).reshape(-1, 1)
phi = nucleo.phi_chapeu(np.array([alpha_de(r)]), R, rf, 5.0)
T = 60
BETAS_ANUAIS = (0.90, 0.93, 0.96, 0.99)
theta = np.array([nucleo.fracoes_consumo(nucleo.recorrencia_A(phi, b ** (1/12), 5.0, T), 5.0)
                  for b in BETAS_ANUAIS])

print('theta_0 por beta anual:', dict(zip(BETAS_ANUAIS, theta[:, 0].round(5))))

theta_0 por beta anual: {0.9: np.float64(0.02141), 0.93: np.float64(0.02109), 0.96: np.float64(0.02079), 0.99: np.float64(0.0205)}


In [8]:
assert np.all(np.diff(theta[:, :T], axis=0) < 0) and np.allclose(theta[:, T], 1.0)

**Teste**: o efeito do prêmio de risco sobre o consumo depende do gamma. Um prêmio maior melhora as oportunidades de investimento, e aí: com gamma > 1 o investidor consome uma fração maior hoje, com gamma < 1 consome uma fração menor e com gamma = 1 os dois se cancelam e o theta_0 não muda. O prêmio começa em zero, onde a carteira ótima é só CDI; abaixo disso a carteira vende a descoberto e as oportunidades também melhoram, então o efeito não seria monotônico.

In [9]:
PREMIOS = (0.0, 0.01, 0.02, 0.03, 0.04, 0.05)          # premio mu - R_f, ao ano
GAMMAS_CONSUMO = (0.5, 1.0, 2.0, 5.0)
rf_anual = rf ** 12 - 1
theta_premio = {}
for g in GAMMAS_CONSUMO:
    th = []
    for p in PREMIOS:
        cen = np.maximum(1 + r + (1 + rf_anual + p) ** (1/12) - 1 - r.mean(), 0).reshape(-1, 1)
        a = nucleo.resolver_alpha_otimo(cen, rf, g)
        A = nucleo.recorrencia_A(nucleo.phi_chapeu(a, cen, rf, g), 0.96 ** (1/12), g, T)
        th.append(nucleo.fracoes_consumo(A, g)[0])
    theta_premio[g] = np.array(th)
    print(f'gamma={g:g}: variacao de theta_0 desde o premio zero (%):', np.round(100 * (theta_premio[g] / th[0] - 1), 2))

gamma=0.5: variacao de theta_0 desde o premio zero (%): [  0.    -0.57  -2.25  -4.97  -8.62 -13.1 ]


gamma=1: variacao de theta_0 desde o premio zero (%): [0. 0. 0. 0. 0. 0.]


gamma=2: variacao de theta_0 desde o premio zero (%): [0.   0.07 0.26 0.59 1.04 1.62]


gamma=5: variacao de theta_0 desde o premio zero (%): [0.   0.04 0.17 0.37 0.66 1.02]


In [10]:
assert np.all(np.diff(theta_premio[0.5]) < 0) and np.allclose(theta_premio[1.0], theta_premio[1.0][0])
assert np.all(np.diff(theta_premio[2.0]) > 0) and np.all(np.diff(theta_premio[5.0]) > 0)